# 🎲 Decoding & Sampling

[Sam Foreman](https://samforeman.me)
(\[[ALCF](https://alcf.anl.gov/about/people/sam-foreman)\](<https://alcf.anl.gov/about/people/sam-foreman>))  
2026-07-29

> **Authors**
>
> Written by [Sam Foreman](https://samforeman.me) for the [Intro to HPC
> Bootcamp](https://intro-hpc-bootcamp.alcf.anl.gov/). Decoding examples
> adapted from Sebastian Raschka’s [*Reasoning From Scratch*,
> ch. 4](https://github.com/rasbt/reasoning-from-scratch).

[](https://colab.research.google.com/github/saforem2/intro-hpc-bootcamp/blob/main/docs/02-llms/5-decoding-and-sampling/index.ipynb)
[](https://github.com/saforem2/intro-hpc-bootcamp/blob/main/content/02-llms/5-decoding-and-sampling/index.qmd)

In [\[2.0\] Intro to LLMs](../0-intro-to-llms/index.qmd) we saw that a
GPT maps a sequence of tokens to a grid of scores: one **probability
distribution over the whole vocabulary** for each position. But how do
we get from that distribution to the *next word*? That choice is called
**decoding**, and it has a huge effect on whether the output is
repetitive and safe, or varied and creative (or unhinged).

> **🎯 What you’ll learn**
>
> - Generation = repeatedly **sampling the next token** from a
>   distribution.
> - **Greedy** decoding (always take the top token) vs **sampling**.
> - **Temperature** — one knob that flattens or sharpens the
>   distribution.
> - **Top-p (nucleus) sampling** — keep only the most probable tokens,
>   then sample.
>
> Everything here runs on a **laptop or Colab** (CPU) with the small
> `gpt2` model. No GPU required.

In [2]:
# On Colab / a fresh env, install transformers + the bootcamp helpers.
try:
    import bootcamp  # noqa: F401
    import transformers  # noqa: F401
except ImportError:
    %pip install -q "git+https://github.com/saforem2/intro-hpc-bootcamp" transformers

In [3]:
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from bootcamp.plotly_theme import apply_theme, COLORS
apply_theme()   # house style + inline plotly.js

torch.manual_seed(0)
tok = GPT2TokenizerFast.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2").eval()

## 🔢 The next-token distribution

A single forward pass gives the model’s scores (**logits**) for every
token in the vocabulary at the next position. Softmax turns those into a
probability distribution. Let’s look at the most likely next tokens for
a prompt:

In [4]:
import plotly.graph_objects as go

prompt = "The capital of France is"
ids = tok(prompt, return_tensors="pt").input_ids
with torch.no_grad():
    logits = model(ids).logits[0, -1]        # logits for the position after the prompt
probs = F.softmax(logits, dim=-1)

topk = torch.topk(probs, 15)
labels = [tok.decode([i]).strip() or "␣" for i in topk.indices]
fig = go.Figure(go.Bar(x=labels, y=topk.values.tolist(),
                       marker_color=COLORS["blue"]))
fig.update_layout(height=340, margin=dict(t=40),
                  title=f"P(next token | {prompt!r})",
                  xaxis_title="candidate token", yaxis_title="probability")
fig.show()

No single token is close to certain: the model is genuinely *unsure*,
and several completions are reasonable. **How we pick from this
distribution is decoding.**

## 🎯 Greedy vs. sampling

The simplest rule is **greedy**: always take the single most probable
token (`argmax`). It’s deterministic (same prompt, same output, every
time), which makes it repetitive and prone to loops.

In [5]:
gen_kwargs = dict(max_new_tokens=20, pad_token_id=tok.eos_token_id)

with torch.no_grad():
    greedy = model.generate(ids, do_sample=False, **gen_kwargs)
print("GREEDY:", tok.decode(greedy[0], skip_special_tokens=True))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.

GREEDY: The capital of France is the capital of the French Republic, and the capital of the French Republic is the capital of the French

**Sampling** (`do_sample=True`) instead *draws* the next token according
to its probability, so more probable tokens appear more often, but the
output varies run to run. Here are three independent samples of the same
prompt:

In [6]:
with torch.no_grad():
    samples = model.generate(ids, do_sample=True, num_return_sequences=3,
                             **gen_kwargs)
for i, s in enumerate(samples):
    print(f"SAMPLE {i}:", tok.decode(s, skip_special_tokens=True))

SAMPLE 0: The capital of France is a big place, the largest city in France so it's not an issue.

This is
SAMPLE 1: The capital of France is struggling to find a replacement for its current mayor. With a population that was 8.8 million in
SAMPLE 2: The capital of France is now occupied by Al Qaeda and now by a French-made satellite, the BMP-8s

Greedy gives you the “safe” path; sampling gives you variety. The next
two knobs control *how much* variety.

## 🌡️ Temperature

**Temperature** `T` rescales the logits *before* the softmax: dividing
by `T`.

- `T < 1` **sharpens** the distribution (more weight on the top tokens →
  closer to greedy, more focused).
- `T > 1` **flattens** it (more weight on the long tail → more diverse,
  riskier).
- `T = 1` leaves it unchanged.

In [7]:
def scale_by_temperature(logits, T):
    """Rescale logits by temperature (T>0). Smaller T -> sharper distribution."""
    return logits / T

Watch what temperature does to the *same* next-token distribution:

In [8]:
fig = go.Figure()
palette = {0.5: COLORS["green"], 1.0: COLORS["blue"], 1.5: COLORS["red"]}
for T, color in palette.items():
    p = F.softmax(scale_by_temperature(logits, T), dim=-1)
    tk = torch.topk(probs, 12).indices          # same 12 tokens for a fair comparison
    fig.add_bar(x=[tok.decode([i]).strip() or "␣" for i in tk],
                y=p[tk].tolist(), name=f"T = {T}", marker_color=color, opacity=0.75)
fig.update_layout(height=360, margin=dict(t=40), barmode="group",
                  title="Temperature reshapes the next-token distribution",
                  xaxis_title="candidate token", yaxis_title="probability")
fig.show()

The same effect shows up in generated text: low temperature is focused
and repetitive, high temperature is varied and can go off the rails:

In [9]:
for T in (0.5, 1.0, 1.5):
    with torch.no_grad():
        out = model.generate(ids, do_sample=True, temperature=T, **gen_kwargs)
    print(f"T={T}:", tok.decode(out[0], skip_special_tokens=True))

T=0.5: The capital of France is not only the capital of France but also the capital of the world. The capital of the world is
T=1.0: The capital of France is the first city to allow refugees from Syria to enter France. The move is an important step as France
T=1.5: The capital of France is now Paris-Tows. These borders were once maintained, in the years between the wars (

## 🎯 Top-p (nucleus) sampling

Temperature reshapes *all* tokens, including absurd ones in the far
tail. **Top-p sampling** (a.k.a. *nucleus* sampling) instead **keeps
only the smallest set of top tokens whose probabilities sum to `p`**,
then samples from just those. It adapts to the distribution: when the
model is confident, the nucleus is tiny; when it’s unsure, the nucleus
is larger.

In [10]:
top_p = 0.9
sorted_probs, sorted_idx = torch.sort(probs, descending=True)
cumsum = torch.cumsum(sorted_probs, dim=-1)
n_show = 20
n_kept = int((cumsum <= top_p).sum().item()) + 1   # include the token that crosses p

fig = go.Figure()
fig.add_bar(x=list(range(n_show)), y=sorted_probs[:n_show].tolist(),
            name="sorted probability", marker_color=COLORS["blue"], opacity=0.6)
fig.add_scatter(x=list(range(n_show)), y=cumsum[:n_show].tolist(), mode="lines+markers",
                name="cumulative sum", line=dict(color=COLORS["orange"]))
fig.add_hline(y=top_p, line=dict(color=COLORS["red"], dash="dash"),
              annotation_text=f"top_p = {top_p}")
fig.add_vline(x=n_kept - 0.5, line=dict(color=COLORS["grey"], dash="dot"),
              annotation_text=f"nucleus = {n_kept} tokens")
fig.update_layout(height=360, margin=dict(t=40),
                  title="Top-p keeps the nucleus of most-probable tokens",
                  xaxis_title="token rank (sorted by probability)", yaxis_title="probability")
fig.show()

So for this prompt the nucleus is tiny: `top_p=0.9` samples from only a
handful of tokens and discards the rest of the 50,257-token vocabulary:

In [11]:
print(f"top_p={top_p}: nucleus = {n_kept} tokens "
      f"(out of {len(probs):,}); the other {len(probs) - n_kept:,} are never sampled")

top_p=0.9: nucleus = 1503 tokens (out of 50,257); the other 48,754 are never sampled

Compare a wide nucleus to a narrow one in generated text:

In [12]:
for p in (0.9, 0.5):
    with torch.no_grad():
        out = model.generate(ids, do_sample=True, top_p=p, **gen_kwargs)
    print(f"top_p={p}:", tok.decode(out[0], skip_special_tokens=True))

top_p=0.9: The capital of France is the French capital, which has over 50 million inhabitants. It has some of the most vibrant markets of
top_p=0.5: The capital of France is now the capital of the European Union. The EU is not the European Union. The EU is the

## 🔑 Key takeaways

- An LLM outputs a **probability distribution** over the next token;
  **decoding** is how you pick from it, one token at a time.
- **Greedy** is deterministic and repetitive; **sampling** adds variety.
- **Temperature** sharpens (`T<1`) or flattens (`T>1`) the whole
  distribution.
- **Top-p** keeps only the most-probable *nucleus* and samples from it,
  adapting to how confident the model is.
- In practice you combine them — e.g. `temperature≈0.7, top_p≈0.9` is a
  common “sensible default” for chat models.

➡️ **Next:** these knobs aren’t just for style. Sampling *many*
completions and combining them can make a model measurably **more
accurate**. That’s **self-consistency**, in [\[3.4\] RL &
Reasoning](../../03-advanced-llms/4-rl-and-reasoning/index.qmd).